# Customer Churn Prediction: Feature Engineering

This notebook prepares the Telco churn dataset for supervised classification. The focus is leakage-safe preprocessing, clear feature selection, train/test splitting, encoding, scaling, and class imbalance awareness.

**Goal:** convert raw customer/account fields into model-ready features while preserving a realistic prediction setup.

## 1. Import Libraries

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Load Data

In [ ]:
DATA_PATH = Path("../data/raw/telco_customer_churn.csv")
df = pd.read_csv(DATA_PATH)

df.shape

In [ ]:
df.head()

## 3. Modeling Objective

The target variable is `Churn`.

- `Churn = Yes`: customer left
- `Churn = No`: customer stayed

This is a binary classification problem. The model should estimate the probability that a customer will churn so the business can prioritize retention actions.

## 4. Leakage-Safe Column Review

| Column | Decision | Reason |
|---|---|---|
| `customerID` | Drop | Identifier only. It should not influence prediction. |
| `Churn` | Target only | This is the outcome we are trying to predict. |
| Customer/account/service/billing fields | Keep | These are available before churn and can support prediction. |

A leakage-safe workflow means the model only receives information that would realistically be known before the churn outcome.

In [ ]:
identifier_columns = ["customerID"]
target_column = "Churn"

feature_columns = [col for col in df.columns if col not in identifier_columns + [target_column]]
feature_columns

## 5. Clean Modeling Fields

`TotalCharges` is stored as text in the raw file. It is converted to numeric and missing values are filled with `0` because the missing rows represent customers with `tenure = 0`.

In [ ]:
df_model = df.copy()

df_model["TotalCharges"] = pd.to_numeric(df_model["TotalCharges"], errors="coerce")
missing_total_charges = df_model["TotalCharges"].isnull().sum()
missing_total_charges

In [ ]:
df_model.loc[df_model["TotalCharges"].isnull(), ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

In [ ]:
df_model["TotalCharges"] = df_model["TotalCharges"].fillna(0)
df_model["ChurnFlag"] = df_model["Churn"].map({"No": 0, "Yes": 1})

df_model[["TotalCharges", "ChurnFlag"]].isnull().sum()

### Cleaning Decision

Filling missing `TotalCharges` with `0` is justified here because those customers have no recorded tenure. In a live business setting, this assumption should be validated with billing logic, but for this dataset it is more appropriate than dropping records or using the mean.

## 6. Candidate Predictive Features

Based on the EDA, the expected high-signal features are:

| Expected importance | Features | Reason |
|---|---|---|
| High | `Contract`, `tenure`, `MonthlyCharges`, `TotalCharges`, `PaymentMethod`, `InternetService`, `TechSupport` | Strong relationship to commitment, lifecycle, price, service experience, and billing behavior. |
| Medium | `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `PaperlessBilling`, `MultipleLines` | Service depth and account behavior may affect retention. |
| Low | `gender` | No strong business reason to expect gender to be a major churn driver. |

Later, model feature importance should be compared against these assumptions.

## 7. Define Features and Target

In [ ]:
X = df_model.drop(columns=["customerID", "Churn", "ChurnFlag"])
y = df_model["ChurnFlag"]

X.shape, y.shape

In [ ]:
y.value_counts(normalize=True).mul(100).round(2)

### Class Imbalance Note

The churn class is the minority class. This affects modeling because a model can achieve acceptable accuracy while failing to identify enough churners.

Model evaluation should therefore include recall, precision, F1 score, ROC-AUC, and the confusion matrix. During training, models that support `class_weight="balanced"` can be tested to reduce bias toward the majority class.

## 8. Numeric and Categorical Feature Groups

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

numeric_features, categorical_features

### Preprocessing Plan

- Numeric features will be scaled with `StandardScaler`.
- Categorical features will be encoded with `OneHotEncoder`.
- Unknown categories will be ignored during prediction so the deployed app is more robust.
- Preprocessing will be wrapped inside a pipeline to avoid train/test leakage.

## 9. Train/Test Split

The split is stratified so the churn/non-churn balance is preserved in both training and test data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
pd.DataFrame(
    {
        "train_%": y_train.value_counts(normalize=True).mul(100).round(2),
        "test_%": y_test.value_counts(normalize=True).mul(100).round(2),
    }
)

### Split Interpretation

The train/test split preserves the churn rate across both sets. This gives a more reliable evaluation because the test data reflects the original class distribution.

## 10. Build Preprocessing Pipeline

The preprocessing object is fit only on the training data. The same fitted transformations are then applied to validation/test data and future app inputs.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

preprocessor

In [ ]:
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

X_train_prepared.shape, X_test_prepared.shape

In [ ]:
encoded_feature_names = preprocessor.get_feature_names_out()
encoded_feature_names[:20], len(encoded_feature_names)

### Encoding Interpretation

One-hot encoding expands categorical fields into model-readable binary indicators. This increases the feature count, but it keeps the categories interpretable for later model inspection.

This setup is also deployment-friendly because the same preprocessing pipeline can be saved with the model and reused in the Streamlit app.

## 11. Feature Engineering Summary

| Step | Decision | Why it matters |
|---|---|---|
| Dropped `customerID` | Removed identifier | Prevents meaningless ID memorization. |
| Encoded `Churn` | Converted Yes/No to 1/0 | Creates supervised classification target. |
| Cleaned `TotalCharges` | Converted to numeric and filled tenure-zero missing values with 0 | Makes revenue field usable without dropping valid customers. |
| Stratified split | Preserved churn balance | Makes evaluation more reliable. |
| Scaled numeric features | Used `StandardScaler` | Helps linear models compare numeric fields fairly. |
| Encoded categoricals | Used `OneHotEncoder` | Makes service/account categories usable for ML. |
| Pipeline approach | Used `ColumnTransformer` | Reduces preprocessing leakage and supports deployment. |

The next notebook should train and compare classification models using this preprocessing setup.